In [15]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
# ---------------------------------------------------
# Examples of using DuckDBSource
# ---------------------------------------------------
# Libraries
import duckdb

from icare_risk.clinphen import DuckDBSource
from icare_risk.clinphen.config.schema import load_schema_from_yaml

# Define paths
schema_cfg = '/app/src/icare_risk/config/icare/schema.yaml'

# Load schem and phenotypes definition
schema = load_schema_from_yaml(schema_cfg)
# Create connection source
source = DuckDBSource(connection=duckdb.connect())

In [23]:
# Default Load: Reads subject, timestamp, and mapped fields (code, value, description)
df_vitals = source.load_domain(
    schema.domains["vitals"],
    subjects=['10001', '10002']
)
display(df_vitals.head(5))

,subject,timestamp,code,value,name,unit
0,10001,2021-12-29,9096706,478.72,respiratory rate,breaths/minute
1,10001,2021-12-29,10933808,564.97,"mean arterial pressure, cuff",millimetres of mercury
2,10001,2021-12-29,11554029,3043.61,"creatinine level, blood",umol/l
3,10001,2021-12-29,14161850,7.33,"ph (poct) level, blood",none
4,10001,2021-12-29,10933816,1417.26,weight measured,kilogram


In [24]:
# Filtering specific codes via DuckDB Pushdown (e.g. Heart Rate & Respiratory Rate only)
df_vitals = source.load_domain(
    schema.domains["vitals"],
    subjects=[10001, 10002],
    codes=[13472364, 9096706]
)
display(df_vitals.head(5))

,subject,timestamp,code,value,name,unit
0,10001,2021-12-29 00:00:00,9096706,478.72,respiratory rate,breaths/minute
1,10001,2021-12-29 20:00:00,9096706,585.28,respiratory rate,breaths/minute
2,10001,2021-12-30 00:00:00,13472364,2702.51,heart rate,none
3,10001,2021-12-30 04:00:00,9096706,68.07,respiratory rate,breaths/minute
4,10001,2021-12-30 12:00:00,9096706,408.94,respiratory rate,breaths/minute


In [30]:
# Requesting an extra column not in standard mapping
df_prescribing = source.load_domain(
    schema.domains["prescribing"],
    columns=[
        "ORDERED_UNIT",
        "THERAPEUTICAL_CLASS"
    ]
)
display(df_prescribing.head(5))

,subject,timestamp,code,value,ORDERED_UNIT,THERAPEUTICAL_CLASS
0,10001,2023-06-16,Amoxicillin,118.90,mg,Antibiotics
1,10001,2022-04-05,Propofol,564.63,ml,Anesthetics
2,10001,2021-04-03,Metformin,308.35,mg,Metabolic agents
3,10001,2020-07-03,Vancomycin,350.36,g,Antibiotics
4,10002,2022-01-20,Lisinopril,774.30,mg,Antihypertensives


In [31]:
# Loading all raw physical columns
df_raw = source.load_domain(
    schema.domains["problems"],
    columns=["*"]
)
display(df_raw.head(5))

,SUBJECT,ENCNTR_ID,PROBLEM_CODE,PROBLEM_DESC,PROBLEM_DT_TM,UPDATE_DT_TM,subject,timestamp,code,description
0,10001,1890016,84114007,heart failure,2021-10-27,2021-11-03,10001,2021-10-27,84114007,heart failure
1,10001,9269749,13645005,copd - chronic obstructive pulmonary disease,2022-09-28,2022-11-06,10001,2022-09-28,13645005,copd - chronic obstructive pulmonary disease
2,10001,6917632,38341003,hypertension,2022-02-10,2022-04-26,10001,2022-02-10,38341003,hypertension
3,10001,7319326,84114007,heart failure,2022-04-17,2022-07-15,10001,2022-04-17,84114007,heart failure
4,10001,2822583,38341003,hypertension,2023-03-17,2023-05-23,10001,2023-03-17,38341003,hypertension


In [45]:
# Libraries
import duckdb

from icare_risk.clinphen import DuckDBSource
from icare_risk.clinphen.registry.registry import load_phenotypes_from_yaml
from icare_risk.clinphen.config.schema import load_schema_from_yaml
from icare_risk.clinphen.engine.runner import FeatureMatrixBuilder

# Define paths
phenotype_cfg = '/app/src/icare_risk/config/icare/phenotypes.yaml'
# Load schem and phenotypes definition
registry = load_phenotypes_from_yaml(phenotype_cfg)
# Extract dependencies dynamically
required_codes, required_cols = registry.get_global_dependencies()


def print_lengths(d, title):
    print(f"\n{title}")
    for key, value in d.items():
        print(key, len(value))

print_lengths(required_codes, title='codes')
print_lengths(required_cols, title='columns')
print(required_cols)

Skipping 'definitions': missing module or function definition.
Skipping 'kim_hx_prior_antibiotics_90d': AttributeError - module 'icare_risk.clinphen.phenotypes.utils' has no attribute 'has_medication_in_window'
Skipping 'tumbarello_hx_recent_abx_90d': AttributeError - module 'icare_risk.phenotypes' has no attribute 'has_medication_in_window'
Skipping 'jones_hx_prior_antibiotics_30d': AttributeError - module 'icare_risk.phenotypes' has no attribute 'has_medication_in_window'
Skipping 'gavaghan_age_ge_65': AttributeError - module 'icare_risk.phenotypes' has no attribute 'derive_age_threshold'
Skipping 'gavaghan_hx_prior_antibiotics_fq_ceph_90d': AttributeError - module 'icare_risk.phenotypes' has no attribute 'has_medication_in_window'
Skipping 'increment_bsi_not_urinary_flag': AttributeError - module 'icare_risk.phenotypes' has no attribute 'derive_bsi_not_urinary'
Skipping 'increment_is_non_ecoli_flag': AttributeError - module 'icare_risk.phenotypes' has no attribute 'derive_is_non_eco

In [46]:
print(required_codes['pathology'])

{'bands', 'wbc', 'PCO2_01'}
